In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
main_path = os.path.join(path, 'Q1_data.csv')
main_df = pd.read_csv(main_path)



In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()
main_df.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()
main_df.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
main_df.describe()

In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(main_df['Delivery_Time'].dropna(), bins=50)
plt.title('delivery_time')
plt.show()

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data
df_clean = main_df.drop(columns=['Order_ID'])
df_clean

In [ ]:
# Task 2: Write your code here: Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
df_clean = df_clean.dropna()
print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 3: Write your code here:Check and remove duplicates if any exist
print("duplicate", df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates()
print("duplicate - after", df_clean.duplicated().sum())


df_clean

In [ ]:
df_clean


In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
categorical_cols = ['Vehicle_Type', 'Time_of_Day', 'Traffic_Level', 'Weather']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_clean)
df_clean

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

In [ ]:
# Task 1: Write your code here:
# Define features (X) and target (y)
feature_cols = ['Distance_km',	'Weather'	,'Traffic_Level'	,'Time_of_Day'	,'Vehicle_Type'	,'Preparation_Time_min'	,'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)


In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

# Use the correct split: KFold OR StratifiedKFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")
# Evaluate using MAE (Mean Absolute Error) ONLY
# Predict and evaluate
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  ${mae:,.2f}")
# Print the averaged score across all folds

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:2. Plot predicted delivery time histogram

# Plot Predictions vs Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)
plt.grid(True)

In [ ]:
# Task Bonus: Write your code here: